# 第3章 Notebook：Gray-Scott 反応拡散

対応章: [`../chapters/03_reaction_diffusion_basic.md`](../chapters/03_reaction_diffusion_basic.md)

この notebook は、卒業研究準備セミナーの数値実験用である。上から順に実行すれば、本文で説明した図を再現できる。設定パラメータは上部のセルにまとめてある。乱数は seed を固定している。

## 1. ライブラリ読み込み

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

## 2. パラメータ設定（ここを変えて実験する）

In [ ]:
Du, Dv = 0.16, 0.08    # diffusion coefficients (Dv < Du)
F_feed, k_kill = 0.035, 0.065  # feed / kill rates
N, STEPS = 128, 4000   # grid size, time steps
print('CFL Du*dt/dx^2 =', Du)   # dt=dx=1 -> must be <= 0.25

## 3. ラプラシアン関数（周期境界）

In [ ]:
def laplacian(Z):
    return (np.roll(Z, 1, 0) + np.roll(Z, -1, 0)
            + np.roll(Z, 1, 1) + np.roll(Z, -1, 1) - 4 * Z)

## 4. 初期条件と時間発展

In [ ]:
def gray_scott(N, steps, Du, Dv, F, k, seed=0):
    rng = np.random.default_rng(seed)
    u = np.ones((N, N)); v = np.zeros((N, N))
    r = N // 10; c = N // 2
    u[c-r:c+r, c-r:c+r] = 0.50
    v[c-r:c+r, c-r:c+r] = 0.25
    u += 0.02 * rng.standard_normal((N, N))
    v += 0.02 * rng.standard_normal((N, N))
    for _ in range(steps):
        uvv = u * v * v
        u += Du * laplacian(u) - uvv + F * (1 - u)
        v += Dv * laplacian(v) + uvv - (F + k) * v
    return u, v

u, v = gray_scott(N, STEPS, Du, Dv, F_feed, k_kill)

## 5. パターンの可視化

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(v, cmap='magma')
plt.title(f'Gray-Scott v  (F={F_feed}, k={k_kill})')
plt.axis('off'); plt.colorbar(fraction=0.046)
plt.tight_layout(); plt.show()

## 6. パラメータ (F, k) を変えた比較

斑点・縞・迷路の違いを見る。比較は計算を軽くするため小さめの格子で行う。

In [ ]:
params = [(0.035, 0.065), (0.055, 0.062), (0.025, 0.055)]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (F, k) in zip(axes, params):
    _, vv = gray_scott(80, 3500, Du, Dv, F, k, seed=1)
    ax.imshow(vv, cmap='magma'); ax.set_title(f'F={F}, k={k}'); ax.axis('off')
plt.tight_layout(); plt.show()

## 7. 課題（自分で変更する）

1. `(F_feed, k_kill)` を変え、斑点／縞／迷路のどれになるか記録せよ。
2. ヘビ模様（細い縞）に近づけるには `Dv/Du` をどう変えればよいか試せ。

In [ ]:
# === 課題セル ===
# 例：縞模様を狙う
_, v_try = gray_scott(96, 3500, Du=0.16, Dv=0.08, F=0.022, k=0.051, seed=2)
plt.figure(figsize=(4.5, 4.5)); plt.imshow(v_try, cmap='magma'); plt.axis('off')
plt.title('your experiment'); plt.tight_layout(); plt.show()